In [0]:
# Databricks notebook source

from uuid import uuid4

from pyspark.sql import functions as F
from pyspark.sql.window import Window


# COMMAND ----------

# Configurações

catalogo = "databricks_cata_managed"

tabela_silver = f"{catalogo}.silver.cotacao_moeda"
tabela_quarentena = f"{catalogo}.audit.cambio_registros_quarentena"

moedas_validas = [
    "USD",
    "EUR",
    "GBP",
    "JPY",
    "CAD",
    "AUD",
]

run_id = str(uuid4())

print(f"Run ID do quality check: {run_id}")


# COMMAND ----------

# Criação da tabela de quarentena

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {tabela_quarentena}
    (
        codigo_moeda        STRING,
        data_cotacao        DATE,
        data_hora_cotacao   TIMESTAMP,
        cotacao_compra      DECIMAL(18,6),
        cotacao_venda       DECIMAL(18,6),
        tipo_boletim        STRING,
        arquivo_origem      STRING,
        _data_ingestao      TIMESTAMP,
        _data_atualizacao   TIMESTAMP,
        motivo_rejeicao     STRING,
        run_id              STRING,
        data_rejeicao       TIMESTAMP
    )
    USING DELTA
    """
)

print(f"Tabela de quarentena disponível: {tabela_quarentena}")


# COMMAND ----------

# Leitura da Silver

df_silver = spark.table(tabela_silver)

quantidade_registros = df_silver.count()

print(f"Quantidade de registros na Silver: {quantidade_registros}")


# COMMAND ----------

# Regra crítica: a Silver não pode estar vazia

if quantidade_registros == 0:
    raise Exception(
        """
QUALITY CHECK REPROVADO

A tabela Silver está vazia.

Possíveis causas:
- a API não retornou registros;
- a Bronze não processou os dados;
- a Silver não recebeu registros.

A Gold não será executada.
"""
    )


# COMMAND ----------

# Chave natural da Silver

chave_natural = [
    "codigo_moeda",
    "data_hora_cotacao",
    "tipo_boletim",
]

janela_duplicidade = Window.partitionBy(*chave_natural)


# COMMAND ----------

# Conta quantas vezes cada chave natural aparece

df_validacao = (
    df_silver
    .withColumn(
        "_quantidade_chave",
        F.count(F.lit(1)).over(janela_duplicidade),
    )
)


# COMMAND ----------

# Aplicação das regras de qualidade

df_validacao = (
    df_validacao
    .withColumn(
        "motivo_rejeicao",
        F.when(
            F.col("codigo_moeda").isNull(),
            F.lit("codigo_moeda não pode ser nulo"),
        )
        .when(
            F.col("data_hora_cotacao").isNull(),
            F.lit("data_hora_cotacao não pode ser nula"),
        )
        .when(
            F.col("cotacao_compra").isNull(),
            F.lit("cotacao_compra não pode ser nula"),
        )
        .when(
            F.col("cotacao_compra") <= 0,
            F.lit("cotacao_compra deve ser maior que zero"),
        )
        .when(
            F.col("cotacao_venda").isNull(),
            F.lit("cotacao_venda não pode ser nula"),
        )
        .when(
            F.col("cotacao_venda") <= 0,
            F.lit("cotacao_venda deve ser maior que zero"),
        )
        .when(
            ~F.col("codigo_moeda").isin(moedas_validas),
            F.lit("codigo_moeda não está na lista configurada"),
        )
        .when(
            F.col("_quantidade_chave") > 1,
            F.lit(
                "duplicidade na chave natural: "
                "codigo_moeda, data_hora_cotacao e tipo_boletim"
            ),
        ),
    )
)


# COMMAND ----------

# Separa os registros rejeitados

df_rejeitados = (
    df_validacao
    .filter(F.col("motivo_rejeicao").isNotNull())
    .withColumn(
        "arquivo_origem",
        F.col("_arquivo_lido"),
    )
    .withColumn(
        "run_id",
        F.lit(run_id),
    )
    .withColumn(
        "data_rejeicao",
        F.current_timestamp(),
    )
    .select(
        "codigo_moeda",
        "data_cotacao",
        "data_hora_cotacao",
        "cotacao_compra",
        "cotacao_venda",
        "tipo_boletim",
        "arquivo_origem",
        "_data_ingestao",
        "_data_atualizacao",
        "motivo_rejeicao",
        "run_id",
        "data_rejeicao",
    )
)


# COMMAND ----------

# Contagem dos resultados

quantidade_rejeitados = df_rejeitados.count()
quantidade_validos = quantidade_registros - quantidade_rejeitados

print("==========================================")
print("RESULTADO DO QUALITY CHECK")
print("==========================================")
print(f"Registros analisados: {quantidade_registros}")
print(f"Registros válidos: {quantidade_validos}")
print(f"Registros rejeitados: {quantidade_rejeitados}")
print("==========================================")


# COMMAND ----------

# Grava os registros inválidos na quarentena

if quantidade_rejeitados > 0:
    (
        df_rejeitados.write
        .format("delta")
        .mode("append")
        .saveAsTable(tabela_quarentena)
    )

    print(
        f"{quantidade_rejeitados} registro(s) gravado(s) em "
        f"{tabela_quarentena}"
    )


# COMMAND ----------

# Falha a tarefa quando existem registros inválidos

if quantidade_rejeitados > 0:
    resumo_rejeicoes = (
        df_rejeitados
        .groupBy("motivo_rejeicao")
        .count()
        .orderBy(F.desc("count"))
        .collect()
    )

    motivos_formatados = "\n".join(
        [
            f"- {linha['motivo_rejeicao']}: "
            f"{linha['count']} registro(s)"
            for linha in resumo_rejeicoes
        ]
    )

    raise Exception(
        f"""
QUALITY CHECK REPROVADO

Registros analisados: {quantidade_registros}
Registros rejeitados: {quantidade_rejeitados}

Motivos encontrados:

{motivos_formatados}

Os registros inválidos foram gravados em:

{tabela_quarentena}

A tarefa Gold não será executada.
"""
    )


# COMMAND ----------

print("QUALITY CHECK APROVADO")
print("Nenhum registro inválido foi encontrado.")
print("A tarefa Gold pode ser executada.")